# Azalyst — Notebook 2: Train Model

**Pre-requisite:** Run `azalyst_1_build_cache.ipynb` first.

**What this does:**
- Reads feature cache from `/kaggle/input/azalyst-feature-cache/` (Kaggle Dataset input)
- Trains XGBoost base model + Meta-labeling model on Year 1+2
- Runs walk-forward simulation on Year 3
- Saves results, charts, and ZIP to `/kaggle/working/azalyst_output/`

**Input dataset to attach:**
Add `YOUR_USERNAME/azalyst-feature-cache` as an input dataset before running.
It will appear at `/kaggle/input/azalyst-feature-cache/`

---
Storage used:
- Input (feature cache): up to 100GB ✓
- Output (models + results): ~500MB ✓  — well under 20GB limit

In [ ]:
# ── Cell 0: Install deps ─────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['xgboost', 'lightgbm', 'sortedcontainers']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Deps installed.')

In [ ]:
# ── Cell 1: Imports + Paths ───────────────────────────────────────────────────
import os, sys, time, json, gc, pickle, warnings, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as sp_stats
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
import psutil
warnings.filterwarnings('ignore')

# ── Feature cache comes from Kaggle Dataset input ─────────────────────────────
# Attach "YOUR_USERNAME/azalyst-feature-cache" as input before running
FEATURE_CACHE_DIR = '/kaggle/input/azalyst-feature-cache'
RESULTS_DIR       = '/kaggle/working/azalyst_output'

for d in [RESULTS_DIR, f'{RESULTS_DIR}/models']:
    os.makedirs(d, exist_ok=True)

cache_files = sorted(Path(FEATURE_CACHE_DIR).glob('*.parquet'))
if not cache_files:
    raise RuntimeError(
        f'No parquet files found at {FEATURE_CACHE_DIR}\n'
        'Did you attach the "azalyst-feature-cache" dataset as input?'
    )

print(f'Feature cache  : {FEATURE_CACHE_DIR}')
print(f'Cache files    : {len(cache_files)} symbols')
print(f'Results dir    : {RESULTS_DIR}')
print(f'RAM available  : {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'Disk free      : {shutil.disk_usage("/kaggle/working").free / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: GPU Setup ─────────────────────────────────────────────────────────
import subprocess, xgboost as xgb

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUDA_DEVICE_ORDER']    = 'PCI_BUS_ID'

try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
    print(r.stdout.decode()[:400])
except: pass

def detect_cuda_api():
    try:
        _x = np.random.rand(1000, 10).astype(np.float32)
        _y = np.random.randint(0, 2, 1000)
        try:
            xgb.XGBClassifier(device='cuda', n_estimators=5, verbosity=0).fit(_x, _y)
            return 'new'
        except: pass
        try:
            xgb.XGBClassifier(tree_method='gpu_hist', n_estimators=5, verbosity=0).fit(_x, _y)
            return 'old'
        except: return None
    except: return None

CUDA_API = detect_cuda_api()
HAS_GPU  = CUDA_API is not None
print(f'XGBoost {xgb.__version__}: CUDA_API={CUDA_API} | HAS_GPU={HAS_GPU}')

def mem_gb(): return psutil.virtual_memory().used / 1e9
def aggressive_gc(): gc.collect()

In [ ]:
# ── Cell 3: Config ────────────────────────────────────────────────────────────
FEATURE_COLS = [
    'ret_1bar','ret_1h','ret_4h','ret_1d','ret_2d','ret_3d','ret_1w',
    'vol_ratio','vol_ret_1h','vol_ret_1d','obv_change','vpt_change','vol_momentum',
    'rvol_1h','rvol_4h','rvol_1d','vol_ratio_1h_1d','atr_norm','parkinson_vol','garman_klass',
    'rsi_14','rsi_6','macd_hist','bb_pos','bb_width','stoch_k','stoch_d','cci_14','adx_14','dmi_diff',
    'vwap_dev','amihud','kyle_lambda','spread_proxy','body_ratio','candle_dir',
    'wick_top','wick_bot','price_accel','skew_1d','kurt_1d','max_ret_4h',
    'wq_alpha001','wq_alpha012','wq_alpha031','wq_alpha098',
    'cs_momentum','cs_reversal','vol_adjusted_mom','trend_consistency',
    'vol_regime','trend_strength','corr_btc_proxy','hurst_exp','fft_strength',
    'frac_diff_close',
]

RETRAIN_WEEKS   = 13       # quarterly
TOP_QUANTILE    = 0.15     # top/bottom 15% for long/short
FEE_RATE        = 0.001    # 0.1% per leg
ROUND_TRIP_FEE  = FEE_RATE * 2
HORIZON_BARS    = 48       # 4H @ 5-min
MAX_TRAIN_ROWS  = 4_000_000
MAX_LOAD_ROWS   = 6_000_000
KAGGLE_RAM_GB   = 30

print(f'[Cell 3] Config loaded.')
print(f'  Features      : {len(FEATURE_COLS)}')
print(f'  Retrain every : {RETRAIN_WEEKS} weeks')
print(f'  Fee round-trip: {ROUND_TRIP_FEE*100:.2f}%')

In [ ]:
# ── Cell 4: Load Feature Store ────────────────────────────────────────────────
print(f'[Cell 4] Loading {len(cache_files)} symbol files from feature cache...')
t0 = time.time()

# Memory guard: stride if total rows would exceed MAX_LOAD_ROWS
_probe      = pd.read_parquet(cache_files[0])
_rows_each  = len(_probe)
_total_est  = _rows_each * len(cache_files)
LOAD_STRIDE = max(1, _total_est // MAX_LOAD_ROWS)
if LOAD_STRIDE > 1:
    print(f'  [MEM GUARD] ~{_total_est:,} est. rows -> LOAD_STRIDE={LOAD_STRIDE}')
del _probe; gc.collect()

frames = []
for fp in cache_files:
    try:
        df = pd.read_parquet(fp)
        # Ensure DatetimeIndex with UTC
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index, unit='ms', utc=True)
        elif df.index.tz is None:
            df.index = df.index.tz_localize('UTC')
        if 'symbol' not in df.columns:
            df['symbol'] = fp.stem
        if LOAD_STRIDE > 1:
            df = df.iloc[::LOAD_STRIDE]
        frames.append(df)
    except Exception as e:
        print(f'  [WARN] {fp.stem}: {e}')

if not frames:
    raise RuntimeError('No feature files loaded.')

ALL_DATA = pd.concat(frames, axis=0).sort_index()
del frames; gc.collect()

print(f'  Pooled    : {len(ALL_DATA):,} rows | {ALL_DATA["symbol"].nunique()} symbols')
print(f'  Date range: {ALL_DATA.index.min().date()} → {ALL_DATA.index.max().date()}')
print(f'  RAM used  : {mem_gb():.1f} GB / {KAGGLE_RAM_GB} GB')
print(f'  Loaded in : {time.time()-t0:.1f}s')

# Validate all feature columns are present
missing = [c for c in FEATURE_COLS if c not in ALL_DATA.columns]
if missing:
    raise ValueError(f'Cache missing columns: {missing}. Re-run Notebook 1.')
print(f'  All {len(FEATURE_COLS)} feature columns present ✓')

In [ ]:
# ── Cell 5: Train / Test Split ────────────────────────────────────────────────
print('[Cell 5] Splitting Year1+2 (train) / Year3 (test)...')

global_min  = ALL_DATA.index.min()
global_max  = ALL_DATA.index.max()
total_span  = global_max - global_min
YEAR2_END   = global_min + (total_span * 2 / 3)
YEAR3_START = YEAR2_END

print(f'  Train (Y1+Y2): {global_min.date()} → {YEAR2_END.date()}')
print(f'  Test  (Y3)  : {YEAR3_START.date()} → {global_max.date()}')

train_df = ALL_DATA[ALL_DATA.index < YEAR2_END].copy()

# Cross-sectional median label
if 'future_ret' in train_df.columns:
    train_df['alpha_label'] = (
        train_df.groupby(train_df.index)['future_ret']
        .transform(lambda x: (x > x.median()).astype(float))
    )
else:
    raise KeyError('future_ret missing from cache. Re-run Notebook 1.')

for c in FEATURE_COLS:
    if c not in train_df.columns: train_df[c] = 0.0

feat_matrix = train_df[FEATURE_COLS].values.astype(np.float32)
label_arr   = train_df['alpha_label'].values.astype(np.float32)
ret_arr     = train_df['future_ret'].values.astype(np.float32)

valid_mask  = np.isfinite(feat_matrix).all(axis=1) & np.isfinite(label_arr)
feat_matrix, label_arr, ret_arr = feat_matrix[valid_mask], label_arr[valid_mask], ret_arr[valid_mask]

if len(feat_matrix) > MAX_TRAIN_ROWS:
    idx = np.sort(np.random.choice(len(feat_matrix), MAX_TRAIN_ROWS, replace=False))
    feat_matrix, label_arr, ret_arr = feat_matrix[idx], label_arr[idx], ret_arr[idx]

del train_df; gc.collect()

X_TRAIN, Y_TRAIN, Y_RET = feat_matrix, label_arr, ret_arr
print(f'  Training matrix : {X_TRAIN.shape[0]:,} rows × {X_TRAIN.shape[1]} features')
print(f'  Label balance   : {Y_TRAIN.mean()*100:.1f}% positive (target ~50%)')

In [ ]:
# ── Cell 6: Model definitions ─────────────────────────────────────────────────

def compute_ic(y_pred, y_true):
    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() < 10: return 0.0
    return float(sp_stats.spearmanr(y_pred[mask], y_true[mask])[0])

def make_xgb_params():
    p = dict(
        n_estimators=1000, learning_rate=0.02, max_depth=6,
        min_child_weight=30, subsample=0.8, colsample_bytree=0.7,
        colsample_bylevel=0.7, reg_alpha=0.1, reg_lambda=1.0,
        eval_metric='auc', early_stopping_rounds=50,
        verbosity=0, random_state=42,
    )
    if CUDA_API == 'new':  p['device'] = 'cuda'
    elif CUDA_API == 'old': p['tree_method'] = 'gpu_hist'
    return p

class PurgedTimeSeriesCV:
    def __init__(self, n_splits=5, gap=48):
        self.n_splits = n_splits; self.gap = gap
    def split(self, X):
        n = len(X); fold_size = n // (self.n_splits + 1)
        for i in range(self.n_splits):
            train_end = (i + 1) * fold_size
            val_start = train_end + self.gap
            val_end   = val_start + fold_size
            if val_end > n: break
            yield np.arange(0, train_end), np.arange(val_start, val_end)

def train_model(X, y, y_ret=None, label=''):
    scaler = RobustScaler()
    Xs = scaler.fit_transform(X)
    cv = PurgedTimeSeriesCV(n_splits=5, gap=48)
    aucs, ics = [], []
    for fold, (tr, val) in enumerate(cv.split(Xs), 1):
        if len(np.unique(y[val])) < 2: continue
        m = xgb.XGBClassifier(**make_xgb_params())
        m.fit(Xs[tr], y[tr], eval_set=[(Xs[val], y[val])], verbose=False)
        probs = m.predict_proba(Xs[val])[:, 1]
        try: aucs.append(roc_auc_score(y[val], probs))
        except: pass
        if y_ret is not None and np.isfinite(y_ret[val]).any():
            ics.append(compute_ic(probs, y_ret[val]))
    mean_auc = float(np.mean(aucs)) if aucs else 0.0
    mean_ic  = float(np.mean(ics))  if ics  else 0.0
    icir     = float(np.mean(ics) / (np.std(ics)+1e-8)) if len(ics) > 1 else 0.0
    final = xgb.XGBClassifier(**make_xgb_params())
    split = int(len(Xs) * 0.9)
    final.fit(Xs[:split], y[:split], eval_set=[(Xs[split:], y[split:])], verbose=False)
    importance = pd.Series(final.feature_importances_, index=FEATURE_COLS, name='importance').sort_values(ascending=False)
    return final, scaler, importance, mean_auc, mean_ic, icir

def train_meta_model(primary_model, primary_scaler, X, y, label='meta'):
    Xs = primary_scaler.transform(X)
    cv = PurgedTimeSeriesCV(n_splits=5, gap=48)
    oos_preds = np.full(len(y), np.nan, dtype=np.float32)
    for fold, (tr, val) in enumerate(cv.split(Xs), 1):
        if len(np.unique(y[val])) < 2: continue
        m_temp = xgb.XGBClassifier(**make_xgb_params())
        m_temp.fit(Xs[tr], y[tr], eval_set=[(Xs[val], y[val])], verbose=False)
        oos_preds[val] = m_temp.predict_proba(Xs[val])[:, 1]
    valid = np.isfinite(oos_preds)
    if valid.sum() < 200:
        print(f'  [{label}] Not enough OOS data ({valid.sum()}) — skipping meta')
        return None, None
    meta_y   = ((oos_preds[valid] >= 0.5).astype(float) == y[valid]).astype(float)
    X_meta   = np.column_stack([Xs[valid], oos_preds[valid]])
    meta_scaler = RobustScaler()
    X_meta_s = meta_scaler.fit_transform(X_meta)
    meta_params = make_xgb_params()
    meta_params.update(n_estimators=500, max_depth=4, min_child_weight=50)
    meta = xgb.XGBClassifier(**meta_params)
    split = int(len(X_meta_s) * 0.9)
    meta.fit(X_meta_s[:split], meta_y[:split], eval_set=[(X_meta_s[split:], meta_y[split:])], verbose=False)
    val_acc = float((meta.predict(X_meta_s[split:]) == meta_y[split:]).mean())
    print(f'  [{label}] Meta accuracy: {val_acc*100:.1f}% on {valid.sum():,} OOS rows')
    return meta, meta_scaler

print('[Cell 6] Model definitions ready.')
print(f'  GPU: {HAS_GPU} (CUDA_API={CUDA_API})')

In [ ]:
# ── Cell 7: Train Base + Meta Model (Year 1+2) ────────────────────────────────
BASE_MODEL_PATH  = f'{RESULTS_DIR}/models/model_base_y1y2.json'
BASE_SCALER_PATH = f'{RESULTS_DIR}/models/scaler_base_y1y2.pkl'
META_MODEL_PATH  = f'{RESULTS_DIR}/models/meta_model_base.pkl'
META_SCALER_PATH = f'{RESULTS_DIR}/models/meta_scaler_base.pkl'

# Base model
if Path(BASE_MODEL_PATH).exists() and Path(BASE_SCALER_PATH).exists():
    print('[Cell 7] Loading cached base model...')
    BASE_MODEL = xgb.XGBClassifier()
    BASE_MODEL.load_model(BASE_MODEL_PATH)
    with open(BASE_SCALER_PATH, 'rb') as f: BASE_SCALER = pickle.load(f)
else:
    print(f'[Cell 7] Training base model on {len(X_TRAIN):,} rows  (GPU={HAS_GPU})...')
    t0 = time.time()
    BASE_MODEL, BASE_SCALER, importance, mean_auc, mean_ic, icir = train_model(
        X_TRAIN, Y_TRAIN, Y_RET, label='base_y1y2'
    )
    BASE_MODEL.save_model(BASE_MODEL_PATH)
    with open(BASE_SCALER_PATH, 'wb') as f: pickle.dump(BASE_SCALER, f)
    importance.to_csv(f'{RESULTS_DIR}/feature_importance_base.csv')
    elapsed = time.time() - t0
    print(f'  AUC (CV)  : {mean_auc:.4f}')
    print(f'  IC  (CV)  : {mean_ic:.4f}')
    print(f'  ICIR      : {icir:.4f}')
    print(f'  Elapsed   : {elapsed:.1f}s')
    print(f'  Top 5 features:')
    print(importance.head(5).to_string())
    with open(f'{RESULTS_DIR}/train_summary.json', 'w') as f:
        json.dump({'mean_auc': round(mean_auc,5), 'mean_ic': round(mean_ic,5),
                   'icir': round(icir,5), 'n_rows': int(len(X_TRAIN)),
                   'elapsed_s': round(elapsed,1), 'use_gpu': HAS_GPU}, f, indent=2)

# Meta model
META_MODEL = META_SCALER = None
if Path(META_MODEL_PATH).exists():
    print('[Cell 7] Loading cached meta model...')
    with open(META_MODEL_PATH, 'rb') as f: META_MODEL = pickle.load(f)
    with open(META_SCALER_PATH, 'rb') as f: META_SCALER = pickle.load(f)
else:
    print('[Cell 7] Training meta-labeling model...')
    META_MODEL, META_SCALER = train_meta_model(
        BASE_MODEL, BASE_SCALER, X_TRAIN, Y_TRAIN, label='meta_base'
    )
    if META_MODEL is not None:
        with open(META_MODEL_PATH, 'wb') as f: pickle.dump(META_MODEL, f)
        with open(META_SCALER_PATH, 'wb') as f: pickle.dump(META_SCALER, f)
        print('  Meta model saved.')

gc.collect()
print(f'[Cell 7] Done. RAM: {mem_gb():.1f} GB')

In [ ]:
# ── Cell 8: Walk-Forward Year 3 ───────────────────────────────────────────────
t0_wf = time.time()
print('[Cell 8] Starting Year 3 walk-forward...')
print(f'  Period   : {YEAR3_START.date()} → {global_max.date()}')
print(f'  Retrain  : every {RETRAIN_WEEKS} weeks')
print(f'  Universe : {ALL_DATA["symbol"].nunique()} symbols')
print(f'  Meta     : {"enabled" if META_MODEL is not None else "disabled"}')
print()

weeks = pd.date_range(start=YEAR3_START, end=global_max, freq='W-MON')
if len(weeks) < 2:
    raise RuntimeError('Not enough Year 3 data. Check date splits.')

current_model  = BASE_MODEL
current_scaler = BASE_SCALER
meta_model     = META_MODEL
meta_scaler    = META_SCALER
retrain_count  = 0

all_trades_list      = []
weekly_summary_list  = []
weekly_returns_hist  = []
prev_longs  = set()
prev_shorts = set()
ic_history  = []

for week_num, (ws, we) in enumerate(zip(weeks[:-1], weeks[1:]), 1):

    week_df = ALL_DATA[(ALL_DATA.index >= ws) & (ALL_DATA.index < we)].copy()
    if len(week_df) < 10: continue

    for c_col in FEATURE_COLS:
        if c_col not in week_df.columns: week_df[c_col] = 0.0
    feat_w  = week_df[FEATURE_COLS].values.astype(np.float32)
    valid_w = np.isfinite(feat_w).all(axis=1)
    if valid_w.sum() < 5: continue

    try:
        feat_scaled = current_scaler.transform(feat_w[valid_w])
        probs_w     = current_model.predict_proba(feat_scaled)[:, 1]
    except Exception as e:
        print(f'  Week {week_num}: predict failed — {e}')
        continue

    valid_idx  = np.where(valid_w)[0]
    week_valid = week_df.iloc[valid_idx].copy()
    week_valid['prob'] = probs_w

    # Meta confidence
    meta_conf_arr = np.ones(len(probs_w), dtype=np.float32)
    if meta_model is not None and meta_scaler is not None:
        try:
            meta_input  = np.column_stack([feat_scaled, probs_w.reshape(-1,1)])
            meta_conf_arr = meta_model.predict_proba(meta_scaler.transform(meta_input))[:, 1]
        except: pass
    week_valid['meta_confidence'] = meta_conf_arr

    # Cross-sectional rank with thin-universe fallback
    def rank_cs(grp):
        n = max(1, int(len(grp) * TOP_QUANTILE))
        grp = grp.copy().sort_values('prob')
        grp['signal'] = 'HOLD'
        if len(grp) >= 4:
            grp.iloc[-n:, grp.columns.get_loc('signal')] = 'BUY'
            grp.iloc[:n,  grp.columns.get_loc('signal')] = 'SELL'
        elif len(grp) >= 1:
            grp['signal'] = grp['prob'].apply(
                lambda p: 'BUY' if p > 0.55 else ('SELL' if p < 0.45 else 'HOLD')
            )
        return grp
    week_valid = week_valid.groupby(week_valid.index, group_keys=False).apply(rank_cs)

    # Aggregate symbol signals
    sym_signals = {}
    for _, row in week_valid[week_valid['signal'] != 'HOLD'].iterrows():
        sym = row.get('symbol', '')
        if sym not in sym_signals:
            sym_signals[sym] = {'signal': row['signal'], 'prob': [], 'ret': [], 'meta': []}
        sym_signals[sym]['prob'].append(float(row['prob']))
        fret = row.get('future_ret', np.nan)
        if np.isfinite(fret): sym_signals[sym]['ret'].append(float(fret))
        sym_signals[sym]['meta'].append(float(row['meta_confidence']))

    # Position-tracked fee simulation
    cur_longs = set(); cur_shorts = set(); trades_this_week = []
    for sym, info in sym_signals.items():
        avg_prob = float(np.mean(info['prob']))
        avg_ret  = float(np.mean(info['ret'])) if info['ret'] else 0.0
        avg_meta = float(np.mean(info['meta']))
        if info['signal'] == 'BUY':
            fee = 0.0 if sym in prev_longs else ROUND_TRIP_FEE
            pnl = (avg_ret - fee) * avg_meta * 100
            cur_longs.add(sym)
        else:
            fee = 0.0 if sym in prev_shorts else ROUND_TRIP_FEE
            pnl = (-avg_ret - fee) * avg_meta * 100
            cur_shorts.add(sym)
        trades_this_week.append({
            'week': week_num, 'week_start': str(ws.date()),
            'symbol': sym, 'signal': info['signal'],
            'pred_prob': round(avg_prob, 5),
            'pnl_percent': round(float(pnl), 4),
            'raw_ret_pct': round(avg_ret * 100, 4),
            'meta_size': round(avg_meta, 4),
        })
    all_trades_list.extend(trades_this_week)

    # Confidence-weighted weekly return
    if trades_this_week:
        sizes = np.array([t['meta_size'] for t in trades_this_week])
        pnls  = np.array([t['pnl_percent'] for t in trades_this_week])
        week_ret = float(np.average(pnls, weights=sizes)) / 100
    else:
        week_ret = 0.0
    weekly_returns_hist.append(week_ret)

    # IC + IC-weighted adjustment
    cs_ic = compute_ic(week_valid['prob'].values,
                       week_valid['future_ret'].fillna(0).values) if 'future_ret' in week_valid.columns else 0.0
    ic_history.append(cs_ic)
    ann_proj = ((1 + week_ret) ** 52 - 1) * 100

    # Quarterly retrain
    did_retrain = False
    if week_num % RETRAIN_WEEKS == 0:
        print(f'  Week {week_num:3d}: QUARTERLY RETRAIN...')
        try:
            rt_df = ALL_DATA[ALL_DATA.index < we].copy()
            for c_col in FEATURE_COLS:
                if c_col not in rt_df.columns: rt_df[c_col] = 0.0
            if 'alpha_label' not in rt_df.columns:
                rt_df['alpha_label'] = rt_df.groupby(rt_df.index)['future_ret'].transform(
                    lambda x: (x > x.median()).astype(float))
            Xr = rt_df[FEATURE_COLS].values.astype(np.float32)
            yr = rt_df['alpha_label'].values.astype(np.float32)
            yr_r = rt_df['future_ret'].values.astype(np.float32) if 'future_ret' in rt_df.columns else None
            vm = np.isfinite(Xr).all(axis=1) & np.isfinite(yr)
            Xr, yr = Xr[vm], yr[vm]
            if yr_r is not None: yr_r = yr_r[vm]
            if len(Xr) > MAX_TRAIN_ROWS:
                idx2 = np.sort(np.random.choice(len(Xr), MAX_TRAIN_ROWS, replace=False))
                Xr, yr = Xr[idx2], yr[idx2]
                if yr_r is not None: yr_r = yr_r[idx2]
            m_new, s_new, imp_new, auc_n, ic_n, icir_n = train_model(Xr, yr, yr_r, label=f'y3_w{week_num:03d}')
            current_model = m_new; current_scaler = s_new
            retrain_count += 1
            m_new.save_model(f'{RESULTS_DIR}/models/model_y3_week{week_num:03d}.json')
            print(f'    AUC={auc_n:.4f}  IC={ic_n:.4f}  ICIR={icir_n:.4f}')
            meta_new, meta_scaler_new = train_meta_model(current_model, current_scaler, Xr, yr, label=f'meta_w{week_num:03d}')
            if meta_new is not None:
                meta_model, meta_scaler = meta_new, meta_scaler_new
            did_retrain = True
            del rt_df, Xr, yr; gc.collect()
        except Exception as e:
            print(f'    Retrain failed: {e}')

    # Turnover
    n_cur = len(cur_longs) + len(cur_shorts)
    n_new = len(cur_longs - prev_longs) + len(cur_shorts - prev_shorts)
    turnover_pct = round(n_new / n_cur * 100, 1) if n_cur > 0 else 100.0
    prev_longs, prev_shorts = cur_longs, cur_shorts

    on_track = week_ret >= ((1.10) ** (1/52) - 1)
    weekly_summary_list.append({
        'week': week_num, 'week_start': str(ws.date()), 'week_end': str(we.date()),
        'n_symbols': int(week_valid['symbol'].nunique()) if 'symbol' in week_valid.columns else 0,
        'n_trades': len(trades_this_week),
        'week_return_pct': round(week_ret * 100, 4),
        'annualised_pct': round(ann_proj, 2),
        'ic': round(cs_ic, 5),
        'turnover_pct': turnover_pct,
        'on_track': on_track,
        'retrained': did_retrain,
    })

    if week_num % 13 == 0:
        print(f'  [Resource] RAM={mem_gb():.1f}GB  Elapsed={( time.time()-t0_wf)/60:.0f}min')
        gc.collect()

    if week_num % 4 == 0 or week_num <= 2:
        rolling = np.mean(weekly_returns_hist[-4:]) * 100 if weekly_returns_hist else 0
        print(f'  Week {week_num:3d} | ret={week_ret*100:+.2f}%  IC={cs_ic:+.4f}  '
              f'4w_avg={rolling:+.2f}%  n={len(trades_this_week)}  TO={turnover_pct:.0f}%')

print(f'\n[Cell 8] Walk-forward complete.')
print(f'  Weeks : {len(weekly_summary_list)} | Trades: {len(all_trades_list)} | Retrains: {retrain_count}')
gc.collect()

In [ ]:
# ── Cell 9: Results + Metrics ─────────────────────────────────────────────────
print('[Cell 9] Saving results...')

if not weekly_summary_list:
    print('[WARN] No weekly results. Did Cell 8 run?')
else:
    weekly_df = pd.DataFrame(weekly_summary_list)
    trades_df = pd.DataFrame(all_trades_list)
    weekly_df.to_csv(f'{RESULTS_DIR}/weekly_summary_year3.csv', index=False)
    trades_df.to_csv(f'{RESULTS_DIR}/all_trades_year3.csv', index=False)

    rets   = weekly_df['week_return_pct'].dropna() / 100
    pnls   = trades_df['pnl_percent'].dropna() / 100 if len(trades_df) > 0 else pd.Series(dtype=float)
    ic_s   = weekly_df['ic'].dropna()
    cum    = (1 + rets).cumprod()
    n_wks  = max(len(rets), 1)
    t_ret  = float(cum.iloc[-1] - 1) if len(cum) > 0 else 0.0
    ann_ret = (1 + t_ret) ** (52 / n_wks) - 1
    sharpe  = float(rets.mean() / (rets.std() + 1e-9) * np.sqrt(52))
    max_dd  = float(((cum - cum.cummax()) / (1 + cum.cummax())).min()) if len(cum) > 0 else 0.0
    win_rt  = float((pnls > 0).mean() * 100) if len(pnls) > 0 else 0.0
    ic_mean = float(ic_s.mean()) if len(ic_s) > 0 else 0.0
    ic_std  = float(ic_s.std())  if len(ic_s) > 0 else 0.0
    icir    = float(ic_mean / (ic_std + 1e-8))
    ic_pos  = float((ic_s > 0).mean() * 100) if len(ic_s) > 0 else 0.0
    avg_to  = float(weekly_df['turnover_pct'].mean()) if 'turnover_pct' in weekly_df.columns else 0.0

    performance = {
        'total_weeks': n_wks, 'total_trades': len(trades_df), 'retrains': retrain_count,
        'total_return_pct': round(t_ret * 100, 2),
        'annualised_pct': round(ann_ret * 100, 2),
        'sharpe': round(sharpe, 4),
        'max_drawdown_pct': round(max_dd * 100, 2),
        'win_rate_pct': round(win_rt, 2),
        'ic_mean': round(ic_mean, 5),
        'icir': round(icir, 4),
        'ic_positive_pct': round(ic_pos, 1),
        'avg_turnover_pct': round(avg_to, 1),
        'meta_labeling': META_MODEL is not None,
        'use_gpu': HAS_GPU,
    }
    with open(f'{RESULTS_DIR}/performance_year3.json', 'w') as f:
        json.dump(performance, f, indent=2)

    print(f'\n  ── Year 3 Out-of-Sample Results ──')
    print(f'  Total Return   : {performance["total_return_pct"]:>8.2f}%')
    print(f'  Annualised     : {performance["annualised_pct"]:>8.2f}%')
    print(f'  Sharpe         : {performance["sharpe"]:>8.4f}')
    print(f'  Max Drawdown   : {performance["max_drawdown_pct"]:>8.2f}%')
    print(f'  Win Rate       : {performance["win_rate_pct"]:>8.2f}%')
    print(f'  IC Mean        : {performance["ic_mean"]:>8.5f}')
    print(f'  ICIR           : {performance["icir"]:>8.4f}')
    print(f'  IC Positive %  : {performance["ic_positive_pct"]:>8.1f}%')
    print(f'  Avg Turnover   : {performance["avg_turnover_pct"]:>8.1f}%')
    print(f'  Retrains       : {retrain_count}')
    print(f'  Meta-labeling  : {performance["meta_labeling"]}')

In [ ]:
# ── Cell 10: Charts + ZIP ─────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import zipfile

print('[Cell 10] Generating charts...')

if weekly_summary_list:
    weekly_df = pd.DataFrame(weekly_summary_list)
    trades_df = pd.DataFrame(all_trades_list)

    fig = plt.figure(figsize=(16, 11))
    fig.suptitle('Azalyst — Year 3 Walk-Forward (XGBoost + Meta-Labeling + Position-Tracked Fees)',
                 fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

    ax1 = fig.add_subplot(gs[0, 0])
    rets = weekly_df['week_return_pct'].fillna(0) / 100
    cum  = ((1 + rets).cumprod() - 1) * 100
    ax1.plot(weekly_df['week'], cum, color='#1f77b4', linewidth=2)
    ax1.fill_between(weekly_df['week'], cum, alpha=0.12, color='#1f77b4')
    ax1.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax1.set_title('Cumulative Return (%)', fontweight='bold')
    ax1.set_xlabel('Week #'); ax1.set_ylabel('%'); ax1.grid(True, alpha=0.25)

    ax2 = fig.add_subplot(gs[0, 1])
    wr = weekly_df['week_return_pct'].dropna()
    ax2.hist(wr, bins=min(30, max(10, len(wr)//3)), color='#ff7f0e', alpha=0.72, edgecolor='black', linewidth=0.4)
    if len(wr) > 2:
        ax2.axvline(wr.mean(),   color='red',   linewidth=1.8, linestyle='--', label=f'Mean {wr.mean():.2f}%')
        ax2.axvline(wr.median(), color='green', linewidth=1.2, linestyle=':',  label=f'Median {wr.median():.2f}%')
        ax2.legend(fontsize=9)
    ax2.set_title('Weekly Return Distribution', fontweight='bold')
    ax2.set_xlabel('Weekly Return (%)'); ax2.set_ylabel('Count'); ax2.grid(True, alpha=0.25)

    ax3 = fig.add_subplot(gs[1, 0])
    ic_vals = weekly_df['ic'].fillna(0)
    ax3.bar(weekly_df['week'], ic_vals,
            color=['#2ca02c' if v > 0 else '#d62728' for v in ic_vals], alpha=0.75, width=0.8)
    if len(ic_vals) > 2:
        ax3.axhline(ic_vals.mean(), color='navy', linewidth=1.5, linestyle='--',
                    label=f'Mean IC {ic_vals.mean():.4f}')
        ax3.legend(fontsize=9)
    ax3.axhline(0, color='black', linewidth=0.6)
    ax3.set_title('Weekly IC (Information Coefficient)', fontweight='bold')
    ax3.set_xlabel('Week #'); ax3.set_ylabel('Spearman IC'); ax3.grid(True, alpha=0.25)

    ax4 = fig.add_subplot(gs[1, 1])
    if len(trades_df) > 0 and 'pnl_percent' in trades_df.columns:
        pnl = trades_df['pnl_percent'].dropna()
        ax4.hist(pnl, bins=min(40, max(10, len(pnl)//20)), color='#9467bd', alpha=0.72,
                 edgecolor='black', linewidth=0.3)
        ax4.axvline(pnl.mean(), color='red', linewidth=1.8, linestyle='--',
                    label=f'Mean {pnl.mean():.3f}%')
        ax4.axvline(0, color='black', linewidth=0.8)
        ax4.legend(fontsize=9)
        ax4.set_title(f'Trade P&L  (n={len(pnl):,})', fontweight='bold')
        ax4.set_xlabel('P&L per Trade (%)'); ax4.set_ylabel('Count'); ax4.grid(True, alpha=0.25)

    chart_path = f'{RESULTS_DIR}/performance_year3.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  Chart saved → {chart_path}')

    # ZIP results
    zip_path = f'{RESULTS_DIR}/azalyst_results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fp in Path(RESULTS_DIR).rglob('*'):
            if fp.is_file() and fp.suffix in ('.csv', '.json', '.png', '.pkl'):
                zf.write(fp, fp.relative_to(Path(RESULTS_DIR).parent))
    zip_mb = Path(zip_path).stat().st_size / 1e6
    print(f'  Results ZIP → {zip_path}  ({zip_mb:.1f} MB)')

    print(f'''
═══════════════════════════════════════════════════
  AZALYST — RUN COMPLETE
═══════════════════════════════════════════════════
  Total Return   : {performance["total_return_pct"]:.2f}%
  Annualised     : {performance["annualised_pct"]:.2f}%
  Sharpe         : {performance["sharpe"]:.4f}
  Max Drawdown   : {performance["max_drawdown_pct"]:.2f}%
  IC Mean        : {performance["ic_mean"]:.5f}
  ICIR           : {performance["icir"]:.4f}
  Meta-labeling  : {performance["meta_labeling"]}
  GPU used       : {HAS_GPU}
═══════════════════════════════════════════════════
  Download azalyst_results.zip from Output tab
''')